<a href="https://colab.research.google.com/github/dhruvi003/ai-engineering-learning/blob/main/embedding_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install sentence-transformers -q

In [2]:
from sentence_transformers import SentenceTransformer

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
sentence = "The pizza was good and tasty"

In [5]:
embedding = model.encode(sentence)

In [7]:
embedding.shape

(384,)

In [10]:
print(type(embedding))
print(embedding[:10].round(4))

<class 'numpy.ndarray'>
[-0.0815  0.1096  0.0237  0.0803 -0.0941 -0.0184 -0.0133 -0.0099 -0.0407
 -0.1172]


In [12]:
print(embedding.min(), embedding.max())

-0.13768393 0.15543757


In [13]:
import numpy as np


In [19]:
def cosine_similarity_manual(a: np.ndarray, b: np.ndarray) -> float:
    """dot(a,b) / (|a| * |b|) — measures the angle between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


In [16]:
pairs = [
    ("dog", "dog"),
    ("dog", "cat"),
    ("teddy", "animal"),
    ("naruto", "luffy"),
    ("avatar", "john")
]

In [21]:
# print(f"{'Pair':<30} Score")
for a,b in pairs:
    ans = cosine_similarity_manual(model.encode(a),model.encode(b))
    print(f"{a:<30} {b:<30} {ans}")



dog                            dog                            1.0
dog                            cat                            0.6606374382972717
teddy                          animal                         0.33553776144981384
naruto                         luffy                          0.2061934620141983
avatar                         john                           0.3521510660648346


In [ ]:
# Cell 4 — build a corpus and embed it in one batch
# this is the "offline ingestion" side of RAG

In [22]:
corpus = [
    # ML / AI
    "Neural networks learn by adjusting weights through backpropagation.",
    "Transformers use self-attention to relate tokens in a sequence.",
    "Gradient descent minimizes a loss function by following the slope.",
    "Overfitting happens when a model memorizes training data instead of generalizing.",
    "A vector database stores embeddings and supports fast similarity search.",
    # Python
    "Python lists are mutable ordered sequences of objects.",
    "Decorators in Python wrap a function to modify its behaviour.",
    "The GIL prevents multiple threads from executing Python bytecode simultaneously.",
    "List comprehensions are a concise way to create lists from iterables.",
    "Type hints in Python improve readability and enable static analysis.",
     # Cooking
    "Maillard reaction causes browning and complex flavours when meat is seared.",
    "Emulsification combines oil and water using an emulsifier like egg yolk.",
    "Sous vide cooking uses precise water temperature to cook food evenly.",
    "Gluten forms when flour proteins hydrate and are worked mechanically.",
    "Resting meat after cooking lets juices redistribute throughout the cut.",
    # Finance
    "Compound interest grows exponentially because you earn interest on interest.",
    "Diversification reduces portfolio risk by spreading investments across assets.",
    "A bear market is a decline of 20% or more from recent highs.",
    "Dollar-cost averaging means investing a fixed amount at regular intervals.",
    "Inflation erodes purchasing power by increasing the price of goods over time.",

]

In [23]:
corpus_embeddings = model.encode(corpus, show_progress_bar=True)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [24]:
corpus_embeddings.shape

(20, 384)

In [25]:
from sklearn.metrics.pairwise import cosine_similarity

In [32]:
# semantic search function
def semantic_search(query:str, corpus:list[str], corpus_embeddings:np.ndarray, top_k:int = 3, min_score:float=0.0) -> list[dict]:
      query_embedding = model.encode(query)

      scores = cosine_similarity([query_embedding], corpus_embeddings)[0]

      top_indices = np.argsort(scores)[::-1][:top_k]
      return [
        {"rank": i+1, "score": round(float(scores[idx]), 4), "text": corpus[idx]}
        for i, idx in enumerate(top_indices)
        if scores[idx] >= min_score
    ]


In [27]:
queries = [
    "How do neural networks actually learn?",
    "What happens when you cook a steak?",
    "How do I avoid losing money investing?",
    "What makes Python slow with threads?",
    "How does an LLM process text?",
    "What is the best way to save money over time?",  # ambiguous — watch this one
]


In [33]:
for query in queries:
    print(f"\nQuery: {query}")
    for r in semantic_search(query, corpus, corpus_embeddings):
        print(f"  #{r['rank']} ({r['score']})  {r['text'][:70]}...")



Query: How do neural networks actually learn?
  #1 (0.761)  Neural networks learn by adjusting weights through backpropagation....
  #2 (0.3578)  Overfitting happens when a model memorizes training data instead of ge...
  #3 (0.3265)  Gradient descent minimizes a loss function by following the slope....

Query: What happens when you cook a steak?
  #1 (0.505)  Resting meat after cooking lets juices redistribute throughout the cut...
  #2 (0.4274)  Maillard reaction causes browning and complex flavours when meat is se...
  #3 (0.261)  Sous vide cooking uses precise water temperature to cook food evenly....

Query: How do I avoid losing money investing?
  #1 (0.4074)  Diversification reduces portfolio risk by spreading investments across...
  #2 (0.2934)  Dollar-cost averaging means investing a fixed amount at regular interv...
  #3 (0.2838)  Compound interest grows exponentially because you earn interest on int...

Query: What makes Python slow with threads?
  #1 (0.6191)  The GIL prev

In [34]:
query = "What is the capital of france?"
for r in semantic_search(query, corpus, corpus_embeddings, top_k=3):
    print(f"  ({r['score']})  {r['text'][:70]}...")

  (0.1477)  Sous vide cooking uses precise water temperature to cook food evenly....
  (0.0978)  Emulsification combines oil and water using an emulsifier like egg yol...
  (0.0697)  A bear market is a decline of 20% or more from recent highs....


In [35]:
print("\nWith min_score=0.5:")
results = semantic_search(query, corpus, corpus_embeddings, top_k=3, min_score=0.5)
if results:
    for r in results: print(f"  ({r['score']})  {r['text'][:70]}...")
else:
    print("  No results above threshold — model would say 'I don't know'")


With min_score=0.5:
  No results above threshold — model would say 'I don't know'


In [36]:
import time

In [37]:
query = "How does attention work in transformer?"

In [38]:
for model_name, desc in [
    ("all-MiniLM-L6-v2",  "Fast / small  — 80 MB,  384 dims"),
    ("all-mpnet-base-v2", "Slow / better — 420 MB, 768 dims"),
]:
    m = SentenceTransformer(model_name)
    t0 = time.time()
    embs = m.encode(corpus)
    q_emb = m.encode(query)
    elapsed = time.time() - t0
    scores = cosine_similarity(q_emb.reshape(1,-1), embs)[0]
    best = np.argsort(scores)[::-1][0]
    print(f"\n{model_name}  ({desc})")
    print(f"  Encode time: {elapsed:.2f}s")
    print(f"  Top result ({scores[best]:.4f}): {corpus[best]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



all-MiniLM-L6-v2  (Fast / small  — 80 MB,  384 dims)
  Encode time: 0.16s
  Top result (0.6023): Transformers use self-attention to relate tokens in a sequence.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


all-mpnet-base-v2  (Slow / better — 420 MB, 768 dims)
  Encode time: 1.18s
  Top result (0.5970): Transformers use self-attention to relate tokens in a sequence.
